# 🐍 Python `datetime` Masterclass

A comprehensive guide to working with dates, times, timezones, and durations in Python.

**Topics covered:**
1. The `datetime` module overview
2. `date`, `time`, and `datetime` objects
3. `timedelta` — arithmetic with time
4. Formatting & parsing: `strftime` / `strptime`
5. Timezones with `zoneinfo` (Python 3.9+) and `pytz`
6. Working with timestamps (Unix epoch)
7. `calendar` module extras
8. `dateutil` for advanced parsing
9. Common real-world patterns
10. Gotchas & best practices

---
## 1. Module Overview

In [1]:
import datetime

# The datetime module contains these key classes:
print(dir(datetime))

['MAXYEAR', 'MINYEAR', 'UTC', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'date', 'datetime', 'datetime_CAPI', 'time', 'timedelta', 'timezone', 'tzinfo']


In [2]:
# Typical imports
from datetime import date, time, datetime, timedelta, timezone, MINYEAR, MAXYEAR
import calendar

print(f"Earliest supported year: {MINYEAR}")
print(f"Latest supported year:   {MAXYEAR}")

Earliest supported year: 1
Latest supported year:   9999


---
## 2. `date` Objects

In [3]:
# Create a date
d = date(2026, 6, 15)
print(f"date: {d}")
print(f"year={d.year}, month={d.month}, day={d.day}")

# Today
today = date.today()
print(f"today: {today}")

# From ISO string
d2 = date.fromisoformat("2025-12-31")
print(f"from ISO: {d2}")

# From ordinal (days since year 1, Jan 1)
d3 = date.fromordinal(738000)
print(f"from ordinal 738000: {d3}")

# Day of week (0=Monday ... 6=Sunday)
print(f"Weekday of {today}: {today.weekday()} ({today.strftime('%A')})")

date: 2026-06-15
year=2026, month=6, day=15
today: 2026-06-15
from ISO: 2025-12-31
from ordinal 738000: 2021-07-29
Weekday of 2026-06-15: 0 (Monday)


---
## 3. `time` Objects

In [4]:
# Create a time (hour, minute, second, microsecond)
t = time(14, 30, 45, 123456)
print(f"time: {t}")
print(f"hour={t.hour}, minute={t.minute}, second={t.second}, microsecond={t.microsecond}")

# Midnight
midnight = time(0, 0, 0)
print(f"midnight: {midnight}")

# time with timezone info
utc = timezone.utc
t_utc = time(9, 0, 0, tzinfo=utc)
print(f"UTC time: {t_utc}")

# ISO format
print(f"ISO: {t.isoformat()}")

time: 14:30:45.123456
hour=14, minute=30, second=45, microsecond=123456
midnight: 00:00:00
UTC time: 09:00:00+00:00
ISO: 14:30:45.123456


---
## 4. `datetime` Objects

In [5]:
# Create a datetime
dt = datetime(2026, 6, 15, 14, 30, 45)
print(f"datetime: {dt}")

# Now (local time, naive)
now = datetime.now()
print(f"now (local, naive): {now}")

# UTC now (aware)
now_utc = datetime.now(timezone.utc)
print(f"now (UTC, aware):   {now_utc}")

# From date + time
d = date(2026, 1, 1)
t = time(12, 0, 0)
dt2 = datetime.combine(d, t)
print(f"combined: {dt2}")

# Extract parts
print(f"date part: {now.date()}")
print(f"time part: {now.time()}")

datetime: 2026-06-15 14:30:45
now (local, naive): 2026-06-15 18:10:08.284321
now (UTC, aware):   2026-06-15 12:40:08.284382+00:00
combined: 2026-01-01 12:00:00
date part: 2026-06-15
time part: 18:10:08.284321


In [6]:
# Naive vs Aware
naive_dt = datetime(2026, 6, 15, 10, 0, 0)
aware_dt = datetime(2026, 6, 15, 10, 0, 0, tzinfo=timezone.utc)

print(f"naive tzinfo: {naive_dt.tzinfo}")   # None
print(f"aware tzinfo: {aware_dt.tzinfo}")   # UTC

# ⚠️ You cannot compare naive and aware datetimes directly
try:
    print(naive_dt < aware_dt)
except TypeError as e:
    print(f"TypeError: {e}")

naive tzinfo: None
aware tzinfo: UTC
TypeError: can't compare offset-naive and offset-aware datetimes


---
## 5. `timedelta` — Date & Time Arithmetic

In [7]:
# Create timedeltas
one_week   = timedelta(weeks=1)
two_days   = timedelta(days=2)
ninety_min = timedelta(minutes=90)
one_hour   = timedelta(hours=1)

print(f"one_week:   {one_week}")
print(f"ninety_min: {ninety_min}")

# Arithmetic
now = datetime.now()
print(f"\nNow:                   {now}")
print(f"Now + 1 week:          {now + one_week}")
print(f"Now - 2 days:          {now - two_days}")
print(f"Now + 90 minutes:      {now + ninety_min}")

one_week:   7 days, 0:00:00
ninety_min: 1:30:00

Now:                   2026-06-15 18:10:12.268600
Now + 1 week:          2026-06-22 18:10:12.268600
Now - 2 days:          2026-06-13 18:10:12.268600
Now + 90 minutes:      2026-06-15 19:40:12.268600


In [8]:
# Difference between two datetimes
dt1 = datetime(2026, 1, 1)
dt2 = datetime(2026, 6, 15)
diff = dt2 - dt1

print(f"Days between Jan 1 and Jun 15: {diff.days} days")
print(f"Total seconds: {diff.total_seconds()}")

# timedelta components
td = timedelta(days=2, hours=5, minutes=30, seconds=15)
print(f"\ntimedelta days:    {td.days}")
print(f"timedelta seconds: {td.seconds}")
print(f"timedelta total:   {td.total_seconds()} seconds")

Days between Jan 1 and Jun 15: 165 days
Total seconds: 14256000.0

timedelta days:    2
timedelta seconds: 19815
timedelta total:   192615.0 seconds


In [9]:
# Countdown example
event = datetime(2027, 1, 1)
remaining = event - datetime.now()
print(f"Days until 2027: {remaining.days} days")

Days until 2027: 199 days


---
## 6. Formatting & Parsing: `strftime` / `strptime`

In [10]:
now = datetime.now()

# strftime — datetime TO string
print(now.strftime("%Y-%m-%d"))               # 2026-06-15
print(now.strftime("%d/%m/%Y %H:%M:%S"))      # 15/06/2026 14:30:45
print(now.strftime("%A, %B %d, %Y"))          # Monday, June 15, 2026
print(now.strftime("%I:%M %p"))               # 02:30 PM
print(now.strftime("%Y-%m-%dT%H:%M:%S"))      # ISO-like
print(now.isoformat())                         # Built-in ISO 8601

2026-06-15
15/06/2026 18:10:27
Monday, June 15, 2026
06:10 PM
2026-06-15T18:10:27
2026-06-15T18:10:27.357440


In [11]:
# Common format codes reference
codes = {
    "%Y": "4-digit year",
    "%y": "2-digit year",
    "%m": "Month 01-12",
    "%B": "Full month name",
    "%b": "Abbreviated month",
    "%d": "Day 01-31",
    "%A": "Full weekday name",
    "%a": "Abbreviated weekday",
    "%H": "Hour 00-23",
    "%I": "Hour 01-12",
    "%M": "Minute 00-59",
    "%S": "Second 00-59",
    "%f": "Microseconds",
    "%p": "AM or PM",
    "%Z": "Timezone name",
    "%z": "UTC offset ±HHMM",
    "%j": "Day of year 001-366",
    "%W": "Week number (Mon start)",
    "%U": "Week number (Sun start)",
}

for code, desc in codes.items():
    print(f"  {code}  →  {desc:30s}  example: {now.strftime(code)}")

  %Y  →  4-digit year                    example: 2026
  %y  →  2-digit year                    example: 26
  %m  →  Month 01-12                     example: 06
  %B  →  Full month name                 example: June
  %b  →  Abbreviated month               example: Jun
  %d  →  Day 01-31                       example: 15
  %A  →  Full weekday name               example: Monday
  %a  →  Abbreviated weekday             example: Mon
  %H  →  Hour 00-23                      example: 18
  %I  →  Hour 01-12                      example: 06
  %M  →  Minute 00-59                    example: 10
  %S  →  Second 00-59                    example: 27
  %f  →  Microseconds                    example: 357440
  %p  →  AM or PM                        example: PM
  %Z  →  Timezone name                   example: 
  %z  →  UTC offset ±HHMM                example: 
  %j  →  Day of year 001-366             example: 166
  %W  →  Week number (Mon start)         example: 24
  %U  →  Week number (Sun start)   

In [12]:
# strptime — string TO datetime
s1 = "2026-06-15 14:30:00"
dt1 = datetime.strptime(s1, "%Y-%m-%d %H:%M:%S")
print(f"Parsed: {dt1}")

s2 = "15/06/2026"
dt2 = datetime.strptime(s2, "%d/%m/%Y")
print(f"Parsed: {dt2}")

s3 = "June 15, 2026 02:30 PM"
dt3 = datetime.strptime(s3, "%B %d, %Y %I:%M %p")
print(f"Parsed: {dt3}")

# fromisoformat (Python 3.7+, enhanced in 3.11)
dt4 = datetime.fromisoformat("2026-06-15T14:30:00")
print(f"From ISO: {dt4}")

Parsed: 2026-06-15 14:30:00
Parsed: 2026-06-15 00:00:00
Parsed: 2026-06-15 14:30:00
From ISO: 2026-06-15 14:30:00


---
## 7. Timezones

In [13]:
# Built-in: UTC
utc = timezone.utc
now_utc = datetime.now(utc)
print(f"UTC: {now_utc}")

# Fixed offsets
ist = timezone(timedelta(hours=5, minutes=30))  # India Standard Time
now_ist = datetime.now(ist)
print(f"IST (+05:30): {now_ist}")

est = timezone(timedelta(hours=-5))  # US Eastern (no DST)
now_est = datetime.now(est)
print(f"EST (-05:00): {now_est}")

UTC: 2026-06-15 12:41:04.753209+00:00
IST (+05:30): 2026-06-15 18:11:04.753429+05:30
EST (-05:00): 2026-06-15 07:41:04.755501-05:00


In [14]:
# zoneinfo (Python 3.9+) — IANA timezone database
from zoneinfo import ZoneInfo, available_timezones

# Create aware datetimes in named timezones
zones = ["UTC", "America/New_York", "Europe/London", "Asia/Kolkata", "Asia/Tokyo"]
now_utc = datetime.now(ZoneInfo("UTC"))

print("Same moment in different timezones:")
for tz_name in zones:
    tz = ZoneInfo(tz_name)
    local = now_utc.astimezone(tz)
    print(f"  {tz_name:25s}: {local.strftime('%Y-%m-%d %H:%M:%S %Z')}")

Same moment in different timezones:
  UTC                      : 2026-06-15 12:41:05 UTC
  America/New_York         : 2026-06-15 08:41:05 EDT
  Europe/London            : 2026-06-15 13:41:05 BST
  Asia/Kolkata             : 2026-06-15 18:11:05 IST
  Asia/Tokyo               : 2026-06-15 21:41:05 JST


In [15]:
# Converting between timezones
ny_tz = ZoneInfo("America/New_York")
tokyo_tz = ZoneInfo("Asia/Tokyo")

meeting_ny = datetime(2026, 6, 15, 9, 0, 0, tzinfo=ny_tz)
meeting_tokyo = meeting_ny.astimezone(tokyo_tz)

print(f"Meeting in New York: {meeting_ny.strftime('%H:%M %Z')}")
print(f"Same meeting Tokyo:  {meeting_tokyo.strftime('%H:%M %Z')}")

# Attaching timezone to naive datetime (localize)
naive = datetime(2026, 6, 15, 12, 0, 0)
aware = naive.replace(tzinfo=ZoneInfo("Europe/London"))
print(f"Localized: {aware}")

Meeting in New York: 09:00 EDT
Same meeting Tokyo:  22:00 JST
Localized: 2026-06-15 12:00:00+01:00


In [16]:
# DST awareness demo
london = ZoneInfo("Europe/London")

winter = datetime(2026, 1, 15, 12, 0, tzinfo=london)  # GMT
summer = datetime(2026, 7, 15, 12, 0, tzinfo=london)  # BST (+1)

print(f"Winter offset: {winter.utcoffset()}")
print(f"Summer offset: {summer.utcoffset()}")
print(f"Winter abbrev: {winter.strftime('%Z')}")
print(f"Summer abbrev: {summer.strftime('%Z')}")

Winter offset: 0:00:00
Summer offset: 1:00:00
Winter abbrev: GMT
Summer abbrev: BST


---
## 8. Unix Timestamps (Epoch)

In [17]:
import time as time_module

# Current Unix timestamp
ts = time_module.time()
print(f"Unix timestamp now: {ts:.3f}")

# datetime → timestamp
dt = datetime(2026, 6, 15, 0, 0, 0, tzinfo=timezone.utc)
ts2 = dt.timestamp()
print(f"timestamp of {dt}: {ts2}")

# timestamp → datetime (local)
dt_local = datetime.fromtimestamp(ts)
print(f"local from timestamp: {dt_local}")

# timestamp → datetime (UTC) — preferred
dt_utc = datetime.fromtimestamp(ts, tz=timezone.utc)
print(f"UTC from timestamp:   {dt_utc}")

Unix timestamp now: 1781527269.554
timestamp of 2026-06-15 00:00:00+00:00: 1781481600.0
local from timestamp: 2026-06-15 18:11:09.553803
UTC from timestamp:   2026-06-15 12:41:09.553803+00:00


---
## 9. The `calendar` Module

In [18]:
import calendar

# Print a month calendar
print(calendar.month(2026, 6))

# Is it a leap year?
print(f"2024 leap? {calendar.isleap(2024)}")
print(f"2026 leap? {calendar.isleap(2026)}")

# Days in a month
_, days_in_june = calendar.monthrange(2026, 6)
print(f"Days in June 2026: {days_in_june}")

# First weekday of month (0=Mon)
first_weekday, _ = calendar.monthrange(2026, 6)
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(f"June 2026 starts on: {day_names[first_weekday]}")

     June 2026
Mo Tu We Th Fr Sa Su
 1  2  3  4  5  6  7
 8  9 10 11 12 13 14
15 16 17 18 19 20 21
22 23 24 25 26 27 28
29 30

2024 leap? True
2026 leap? False
Days in June 2026: 30
June 2026 starts on: Monday


In [19]:
# Iterate over weeks in a month
print("Weeks in June 2026 (0=Mon):")
for week in calendar.monthcalendar(2026, 6):
    print(f"  {week}")

# All month names and abbrevs
print("\nMonth names:", list(calendar.month_name)[1:])
print("Abbreviated:", list(calendar.month_abbr)[1:])

Weeks in June 2026 (0=Mon):
  [1, 2, 3, 4, 5, 6, 7]
  [8, 9, 10, 11, 12, 13, 14]
  [15, 16, 17, 18, 19, 20, 21]
  [22, 23, 24, 25, 26, 27, 28]
  [29, 30, 0, 0, 0, 0, 0]

Month names: ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
Abbreviated: ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


---
## 10. `dateutil` — Advanced Parsing & Relative Deltas

In [20]:
# Install if needed:
# !pip install python-dateutil

from dateutil import parser as dateutil_parser
from dateutil.relativedelta import relativedelta

# Flexible parsing — handles many formats automatically
samples = [
    "June 15, 2026",
    "15-Jun-2026",
    "2026/06/15 14:30:00",
    "Mon, 15 Jun 2026 14:30:00 +0000",  # RFC 2822
    "15th June 2026",
]

for s in samples:
    try:
        parsed = dateutil_parser.parse(s)
        print(f"  {s!r:40s} → {parsed}")
    except Exception as e:
        print(f"  {s!r:40s} → ERROR: {e}")

  'June 15, 2026'                          → 2026-06-15 00:00:00
  '15-Jun-2026'                            → 2026-06-15 00:00:00
  '2026/06/15 14:30:00'                    → 2026-06-15 14:30:00
  'Mon, 15 Jun 2026 14:30:00 +0000'        → 2026-06-15 14:30:00+00:00
  '15th June 2026'                         → 2026-06-15 00:00:00


In [21]:
# relativedelta — calendar-aware deltas
from dateutil.relativedelta import relativedelta

now = datetime.now()
print(f"Now:                  {now.date()}")
print(f"+ 1 month:            {(now + relativedelta(months=1)).date()}")
print(f"+ 3 months, 2 weeks:  {(now + relativedelta(months=3, weeks=2)).date()}")
print(f"+ 1 year:             {(now + relativedelta(years=1)).date()}")
print(f"- 6 months:           {(now - relativedelta(months=6)).date()}")

# Difference in human-readable terms
birth = datetime(1990, 7, 20)
age = relativedelta(now, birth)
print(f"\nAge: {age.years} years, {age.months} months, {age.days} days")

Now:                  2026-06-15
+ 1 month:            2026-07-15
+ 3 months, 2 weeks:  2026-09-29
+ 1 year:             2027-06-15
- 6 months:           2025-12-15

Age: 35 years, 10 months, 26 days


---
## 11. Real-World Patterns

In [22]:
# ── Pattern 1: Get start/end of day ──
today = date.today()
start_of_day = datetime.combine(today, time.min)  # 00:00:00.000000
end_of_day   = datetime.combine(today, time.max)  # 23:59:59.999999
print(f"Start of day: {start_of_day}")
print(f"End of day:   {end_of_day}")

Start of day: 2026-06-15 00:00:00
End of day:   2026-06-15 23:59:59.999999


In [23]:
# ── Pattern 2: First/last day of month ──
import calendar

def first_last_day(year, month):
    first = date(year, month, 1)
    _, last_day = calendar.monthrange(year, month)
    last = date(year, month, last_day)
    return first, last

first, last = first_last_day(2026, 6)
print(f"June 2026: {first} to {last}")

first, last = first_last_day(2026, 2)
print(f"Feb 2026:  {first} to {last}")

June 2026: 2026-06-01 to 2026-06-30
Feb 2026:  2026-02-01 to 2026-02-28


In [24]:
# ── Pattern 3: Generate a date range ──
def date_range(start: date, end: date):
    """Yield dates from start to end (inclusive)."""
    current = start
    while current <= end:
        yield current
        current += timedelta(days=1)

start = date(2026, 6, 10)
end   = date(2026, 6, 15)
print("Date range:")
for d in date_range(start, end):
    print(f"  {d} ({d.strftime('%A')})")

Date range:
  2026-06-10 (Wednesday)
  2026-06-11 (Thursday)
  2026-06-12 (Friday)
  2026-06-13 (Saturday)
  2026-06-14 (Sunday)
  2026-06-15 (Monday)


In [25]:
# ── Pattern 4: Next occurrence of a weekday ──
def next_weekday(d: date, weekday: int) -> date:
    """Return the next date with the given weekday (0=Mon..6=Sun)."""
    days_ahead = weekday - d.weekday()
    if days_ahead <= 0:  # already passed this week
        days_ahead += 7
    return d + timedelta(days=days_ahead)

today = date.today()
next_friday = next_weekday(today, 4)  # 4 = Friday
next_monday = next_weekday(today, 0)
print(f"Today:        {today} ({today.strftime('%A')})")
print(f"Next Friday:  {next_friday}")
print(f"Next Monday:  {next_monday}")

Today:        2026-06-15 (Monday)
Next Friday:  2026-06-19
Next Monday:  2026-06-22


In [26]:
# ── Pattern 5: Age calculator ──
def calculate_age(birthdate: date) -> dict:
    today = date.today()
    rd = relativedelta(today, birthdate)
    days_old = (today - birthdate).days
    return {
        "years": rd.years,
        "months": rd.months,
        "days": rd.days,
        "total_days": days_old,
    }

birthday = date(1990, 3, 15)
age = calculate_age(birthday)
print(f"Age: {age['years']} years, {age['months']} months, {age['days']} days")
print(f"     ({age['total_days']:,} days old)")

Age: 36 years, 3 months, 0 days
     (13,241 days old)


In [27]:
# ── Pattern 6: Human-friendly elapsed time ──
def time_ago(dt: datetime) -> str:
    """Return a human-readable string like '3 hours ago'."""
    now = datetime.now()
    diff = now - dt
    total_seconds = int(diff.total_seconds())

    if total_seconds < 60:
        return f"{total_seconds} second{'s' if total_seconds != 1 else ''} ago"
    elif total_seconds < 3600:
        m = total_seconds // 60
        return f"{m} minute{'s' if m != 1 else ''} ago"
    elif total_seconds < 86400:
        h = total_seconds // 3600
        return f"{h} hour{'s' if h != 1 else ''} ago"
    else:
        d = total_seconds // 86400
        return f"{d} day{'s' if d != 1 else ''} ago"

# Test
print(time_ago(datetime.now() - timedelta(seconds=45)))
print(time_ago(datetime.now() - timedelta(minutes=20)))
print(time_ago(datetime.now() - timedelta(hours=3)))
print(time_ago(datetime.now() - timedelta(days=5)))

45 seconds ago
20 minutes ago
3 hours ago
5 days ago


In [28]:
# ── Pattern 7: Benchmarking with datetime ──
import time as time_module

start = time_module.perf_counter()

# Simulate some work
total = sum(i**2 for i in range(1_000_000))

elapsed = time_module.perf_counter() - start
print(f"Computed sum in {elapsed:.4f} seconds")
print(f"Result: {total:,}")

Computed sum in 0.0614 seconds
Result: 333,332,833,333,500,000


---
## 12. Gotchas & Best Practices

In [29]:
# ⚠️ GOTCHA 1: datetime.now() is LOCAL and NAIVE
# Always use timezone-aware datetimes in production

# ❌ Avoid
naive_now = datetime.now()

# ✅ Prefer
aware_now = datetime.now(timezone.utc)
# or
aware_now2 = datetime.now(ZoneInfo("UTC"))

print(f"Naive: {naive_now} (tzinfo={naive_now.tzinfo})")
print(f"Aware: {aware_now} (tzinfo={aware_now.tzinfo})")

Naive: 2026-06-15 18:11:27.792887 (tzinfo=None)
Aware: 2026-06-15 12:41:27.792911+00:00 (tzinfo=UTC)


In [30]:
# ⚠️ GOTCHA 2: replace() doesn't convert, it stamps
naive = datetime(2026, 6, 15, 12, 0, 0)

# ❌ Wrong way to "convert" to UTC
wrong = naive.replace(tzinfo=timezone.utc)  # Just stamps UTC, doesn't convert

# ✅ Correct: localize first, then convert
local_aware = naive.replace(tzinfo=ZoneInfo("America/New_York"))
as_utc = local_aware.astimezone(timezone.utc)
print(f"Localized NY: {local_aware}")
print(f"As UTC:       {as_utc}")

Localized NY: 2026-06-15 12:00:00-04:00
As UTC:       2026-06-15 16:00:00+00:00


In [31]:
# ⚠️ GOTCHA 3: timedelta doesn't know months/years
# Adding months is tricky with timedelta

jan31 = date(2026, 1, 31)

# ❌ Can't do: jan31 + timedelta(months=1)
# ✅ Use relativedelta instead:
from dateutil.relativedelta import relativedelta
feb_result = jan31 + relativedelta(months=1)
print(f"Jan 31 + 1 month = {feb_result}")  # Feb 28 (clamped)

Jan 31 + 1 month = 2026-02-28


In [32]:
# ⚠️ GOTCHA 4: Mutable default arguments with dates
# Never use mutable defaults in function signatures

# ❌ Wrong
# def log_event(message, dt=datetime.now()):  # evaluated ONCE at import time!
#     print(f"{dt}: {message}")

# ✅ Correct
def log_event(message, dt=None):
    if dt is None:
        dt = datetime.now()
    print(f"{dt.strftime('%Y-%m-%d %H:%M:%S')}: {message}")

log_event("Server started")
log_event("Manual event", datetime(2026, 1, 1, 0, 0, 0))

2026-06-15 18:11:34: Server started
2026-01-01 00:00:00: Manual event


In [33]:
# ✅ BEST PRACTICE: Store everything in UTC, display in local time
def store_event(name: str) -> dict:
    """Store event with UTC timestamp."""
    return {
        "name": name,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

def display_event(event: dict, tz_name: str = "Asia/Kolkata"):
    """Display event in local timezone."""
    utc_dt = datetime.fromisoformat(event["created_at"])
    local_dt = utc_dt.astimezone(ZoneInfo(tz_name))
    print(f"Event '{event['name']}' at {local_dt.strftime('%Y-%m-%d %H:%M %Z')}")

event = store_event("User signup")
display_event(event, "Asia/Kolkata")
display_event(event, "America/New_York")
display_event(event, "Europe/Paris")

Event 'User signup' at 2026-06-15 18:11 IST
Event 'User signup' at 2026-06-15 08:41 EDT
Event 'User signup' at 2026-06-15 14:41 CEST


---
## 13. Quick Reference Cheat Sheet

| Task | Code |
|------|------|
| Today's date | `date.today()` |
| Current UTC datetime | `datetime.now(timezone.utc)` |
| Format datetime | `dt.strftime("%Y-%m-%d %H:%M:%S")` |
| Parse string | `datetime.strptime(s, fmt)` |
| Add N days | `dt + timedelta(days=N)` |
| Add N months | `dt + relativedelta(months=N)` |
| Difference in days | `(dt2 - dt1).days` |
| Convert timezone | `dt.astimezone(ZoneInfo("TZ"))` |
| Unix timestamp | `dt.timestamp()` |
| From Unix timestamp | `datetime.fromtimestamp(ts, tz=timezone.utc)` |
| Start of day | `datetime.combine(date.today(), time.min)` |
| ISO format | `dt.isoformat()` |
| Is leap year | `calendar.isleap(year)` |
| Days in month | `calendar.monthrange(y, m)[1]` |